## WP016 — Over/under 2.5: does the model beat, or add to, Pinnacle on totals?

See `README.md`. **Primary (pre-declared), model = `baseline`, both held-out sets pooled:** Q1 (leave-one-window-out log loss, Pinnacle-close logit plus model logit minus Pinnacle-close logit alone; hit needs CI < 0 and both halves negative) and Q2 (CLV of model-selected bets at Pinnacle's opening over/under odds, `tau = 0.02`, against Pinnacle's closing fair prices; hit needs CI > 0 and both halves positive). Everything else is exploratory. No sampling.

In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from football_model.evaluation import market as mk
from football_model.model.predict import dc_over_prob

REPO = Path('/Users/hadiahmed/Documents/projects/football-predictor')
WP001 = REPO / 'work_products' / 'wp001_walkforward_cv_baseline'
WP003 = REPO / 'work_products' / 'wp003_bookmaker_benchmark'
WP014 = REPO / 'work_products' / 'wp014_continuity_confirmation'
N_BOOT, TAU, LINE = 5000, 0.02, 2.5
TAUS = [-np.inf, 0.0, 0.01, 0.02, 0.03, 0.05]

def show(df, digits=4):
    print(df.round(digits).to_string(index=False))

shared_old = pickle.load(open(WP001 / 'cv_shared_data.pkl', 'rb'))
shared_new = pickle.load(open(WP014 / 'cv_shared_data.pkl', 'rb'))
df_cv = shared_old['df_cv']
odds = pd.read_pickle(WP003 / 'odds_raw.pkl').dropna(subset=['Date', 'FTR']).reset_index(drop=True)

OU_COLS = ['P>2.5', 'P<2.5', 'PC>2.5', 'PC<2.5']          # Pinnacle pre-closing / closing, over and under

def load_set(arm_ckpt_path, windows, tag, window_offset):
    ckpt = pickle.load(open(arm_ckpt_path, 'rb'))
    assert len(ckpt['results']) == len(windows), (tag, len(ckpt['results']), len(windows))
    fx = mk.model_fixtures(df_cv, windows, ckpt)
    fx['window'] = fx['window'] + window_offset            # keep window ids distinct across the two sets
    J = mk.join_odds(fx, odds, required_cols=OU_COLS)      # raises if a score disagrees with the odds file
    J['set'] = tag
    return fx, J

fx_old, J_old = load_set(WP001 / 'cv_checkpoint.pkl', shared_old['windows'], 'old (WP001, 1 round ahead)', 0)
fx_new, J_new = load_set(WP014 / 'cv_checkpoint_baseline.pkl', shared_new['windows'], 'new (WP014, 1-3 rounds ahead)', 100)
print(f'old: {len(fx_old)} matches, {len(J_old)} with Pinnacle O/U odds;  new: {len(fx_new)} matches, {len(J_new)} with Pinnacle O/U odds')
J_all = pd.concat([J_old, J_new], ignore_index=True)

def over_prob(J):
    rho = [None if pd.isna(r) else float(r) for r in J['rho_dc']]
    return np.array([dc_over_prob(lh, la, rho=r, line=LINE) for lh, la, r in zip(J['lambda_home'], J['lambda_away'], rho)])

old: 401 matches, 360 with Pinnacle O/U odds;  new: 1086 matches, 995 with Pinnacle O/U odds


## Setup checks

The model's over-2.5 probability, Pinnacle's, and the realised over rate. The model's mean goal rate has been below the actual goals per match in earlier work, so a bias in totals would show here.

In [2]:
def prep(J):
    d = {}
    d['over'] = ((J['goals_home'] + J['goals_away']) > LINE).astype(float).to_numpy()
    d['y'] = np.column_stack([d['over'], 1 - d['over']])                        # [over, under]
    d['p_mod_over'] = over_prob(J)
    d['p_mod'] = np.column_stack([d['p_mod_over'], 1 - d['p_mod_over']])
    d['open_odds'] = J[['P>2.5', 'P<2.5']].to_numpy(float)
    d['p_close'] = mk.devig(J[['PC>2.5', 'PC<2.5']].to_numpy(float))
    d['p_open'] = mk.devig(d['open_odds'])
    return d

D = prep(J_all)
print(f"n = {len(J_all)}   actual over-2.5 rate {D['over'].mean():.3f}   Pinnacle close mean P(over) {D['p_close'][:, 0].mean():.3f}   "
      f"model mean P(over) {D['p_mod_over'].mean():.3f}")
print(f"mean total goals: actual {(J_all['goals_home'] + J_all['goals_away']).mean():.3f}   model {(J_all['lambda_home'] + J_all['lambda_away']).mean():.3f}")

n = 1355   actual over-2.5 rate 0.565   Pinnacle close mean P(over) 0.552   model mean P(over) 0.525
mean total goals: actual 2.953   model 2.800


## The tests

`run_tests(J, D, label)` computes the blend sweep, Q1 and Q2 with CLV by threshold for any subset, so the pooled (primary) and the old/new splits use identical code.

In [3]:
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def verdict(lo, hi, h1, h2, favourable):
    ok = (hi < 0 and h1 < 0 and h2 < 0) if favourable == 'negative' else (lo > 0 and h1 > 0 and h2 > 0)
    return 'HIT' if ok else 'no hit'

def run_tests(J, D, label, verbose=True):
    first, second = mk.half_masks(J['date'])
    y_idx = D['over'].astype(int)
    ll = lambda p: -np.log(np.where(y_idx == 1, p, 1 - p))
    p_c, p_m = D['p_close'][:, 0], D['p_mod_over']

    if verbose:
        print(f'=========== {label} (n = {len(J)}) ===========')
        print(f'log loss: Pinnacle close {ll(p_c).mean():.4f}   Pinnacle open {ll(D["p_open"][:, 0]).mean():.4f}   model {ll(p_m).mean():.4f}')
        print('\nblend (1-w)*Pinnacle_close + w*model, paired log-loss vs pure Pinnacle:')
        rows = []
        for w in [0.0, 0.05, 0.10, 0.20, 0.30, 0.50, 1.0]:
            d = ll((1 - w) * p_c + w * p_m) - ll(p_c)
            m, lo, hi = mk.bootstrap_ci(d, N_BOOT); rows.append({'w_model': w, 'diff_vs_pinnacle': m, 'lo': lo, 'hi': hi})
        show(pd.DataFrame(rows), 5)

    # Q1
    groups = J['window'].to_numpy()
    d1 = mk.leave_group_out_logloss(np.column_stack([logit(p_c), logit(p_m)]), y_idx, groups) \
         - mk.leave_group_out_logloss(logit(p_c)[:, None], y_idx, groups)
    m1, lo1, hi1 = mk.bootstrap_ci(d1, N_BOOT)
    h11, h12 = d1[first].mean(), d1[second].mean()

    # Q2
    mask = mk.edge(D['p_mod'], D['open_odds']) > TAU
    full = mk.clv_table(D['open_odds'], D['p_close'], mask, D['y'], n_boot=N_BOOT)
    halves = [mk.clv_table(D['open_odds'][h], D['p_close'][h], mask[h], n_boot=N_BOOT)['clv'] for h in (first, second)]
    if verbose:
        print('\nCLV of model-selected bets at Pinnacle opening odds (any side, over or under):')
        rows = [{'tau': t, **mk.clv_table(D['open_odds'], D['p_close'], mk.edge(D['p_mod'], D['open_odds']) > t, D['y'], n_boot=N_BOOT)} for t in TAUS]
        show(pd.DataFrame(rows), 4)
    return pd.DataFrame([
        {'test': 'Q1 log loss, Pinnacle+model - Pinnacle alone', 'n': len(d1), 'estimate': m1, 'lo': lo1, 'hi': hi1,
         'half_1': h11, 'half_2': h12, 'result': verdict(lo1, hi1, h11, h12, 'negative')},
        {'test': f'Q2 CLV per bet, tau={TAU}', 'n': full['n_bets'], 'estimate': full['clv'], 'lo': full['clv_lo'], 'hi': full['clv_hi'],
         'half_1': halves[0], 'half_2': halves[1], 'result': verdict(full['clv_lo'], full['clv_hi'], halves[0], halves[1], 'positive')},
    ])

summary = run_tests(J_all, D, 'POOLED, baseline (PRIMARY)')
print('\n----- primary verdicts -----')
show(summary, 5)

=========== POOLED, baseline (PRIMARY) (n = 1355) ===========
log loss: Pinnacle close 0.6711   Pinnacle open 0.6725   model 0.6838

blend (1-w)*Pinnacle_close + w*model, paired log-loss vs pure Pinnacle:


 w_model  diff_vs_pinnacle       lo      hi
    0.00           0.00000  0.00000 0.00000
    0.05           0.00014 -0.00026 0.00051
    0.10           0.00033 -0.00047 0.00108
    0.20           0.00086 -0.00073 0.00236
    0.30           0.00160 -0.00078 0.00385
    0.50           0.00369 -0.00024 0.00745
    1.00           0.01262  0.00472 0.02019



CLV of model-selected bets at Pinnacle opening odds (any side, over or under):


 tau  n_bets     clv  clv_lo  clv_hi     roi  roi_lo  roi_hi
-inf    2710 -0.0318 -0.0324 -0.0312 -0.0347 -0.0443 -0.0240
0.00    1072 -0.0303 -0.0338 -0.0270 -0.0263 -0.0900  0.0400
0.01     995 -0.0298 -0.0334 -0.0262 -0.0175 -0.0847  0.0520
0.02     931 -0.0298 -0.0336 -0.0261 -0.0266 -0.0957  0.0457
0.03     850 -0.0291 -0.0332 -0.0252 -0.0323 -0.1063  0.0445
0.05     718 -0.0286 -0.0331 -0.0242 -0.0429 -0.1227  0.0376

----- primary verdicts -----
                                        test    n  estimate       lo       hi   half_1   half_2 result
Q1 log loss, Pinnacle+model - Pinnacle alone 1355   0.00097  0.00059  0.00136  0.00123  0.00072 no hit
                    Q2 CLV per bet, tau=0.02  931  -0.02977 -0.03356 -0.02608 -0.02775 -0.03184 no hit


### Exploratory: old and new sets separately

Same tests on each held-out set on its own: a consistency check, not an extra chance at a hit.

In [4]:
for tag, J in J_all.groupby('set'):
    idx = J.index.to_numpy()
    Dsub = {k: v[idx] for k, v in D.items()}
    print(f'--- {tag} ---')
    show(run_tests(J.reset_index(drop=True), Dsub, tag, verbose=False), 5)
    print()

--- new (WP014, 1-3 rounds ahead) ---
                                        test   n  estimate       lo       hi   half_1   half_2 result
Q1 log loss, Pinnacle+model - Pinnacle alone 995   0.00132 -0.00003  0.00268  0.00202  0.00063 no hit
                    Q2 CLV per bet, tau=0.02 669  -0.03175 -0.03634 -0.02737 -0.02595 -0.03781 no hit

--- old (WP001, 1 round ahead) ---
                                        test   n  estimate       lo       hi   half_1   half_2 result
Q1 log loss, Pinnacle+model - Pinnacle alone 360   0.00177 -0.00194  0.00559  0.00253  0.00104 no hit
                    Q2 CLV per bet, tau=0.02 262  -0.02472 -0.03203 -0.01753 -0.03325 -0.01694 no hit



### Exploratory: model-free soft-vs-sharp on over/under (all matches with Pinnacle O/U odds)

Bet a soft book's closing over/under price whenever Pinnacle's de-vigged closing probability says it is worth more than `tau`, settled at the soft book's price. `-inf` bets every outcome (control: about minus the margin). No model involved.

In [5]:
odds_ou = odds.dropna(subset=['PC>2.5', 'PC<2.5']).reset_index(drop=True)
pf = mk.devig(odds_ou[['PC>2.5', 'PC<2.5']].to_numpy(float))
yy = (odds_ou['FTHG'] + odds_ou['FTAG'] > LINE).astype(float).to_numpy()
y2 = np.column_stack([yy, 1 - yy])
for name, cols in [('Bet365 closing', ['B365C>2.5', 'B365C<2.5']), ('Best available closing (Max)', ['MaxC>2.5', 'MaxC<2.5'])]:
    book = odds_ou[cols].to_numpy(float)
    ok = ~np.isnan(book).any(axis=1)
    print(f'--- {name} vs Pinnacle fair ({ok.sum()} matches) ---')
    show(mk.roi_table(pf[ok], book[ok], y2[ok], TAUS, n_boot=N_BOOT), 4)
    print()

--- Bet365 closing vs Pinnacle fair (2099 matches) ---


 tau  n_bets  n_matches     roi      lo      hi  claimed_edge  hit_rate
-inf    4198       2099 -0.0473 -0.0557 -0.0387       -0.0476    0.5000
0.00      75         75 -0.0903 -0.3728  0.2006        0.0409    0.3600
0.01      51         51 -0.2484 -0.5961  0.1166        0.0578    0.2745
0.02      33         33 -0.4267 -0.8374  0.0462        0.0811    0.1818
0.03      26         26 -0.2723 -0.7866  0.3201        0.0964    0.2308
0.05      17         17  0.0029 -0.7333  0.8750        0.1285    0.2941

--- Best available closing (Max) vs Pinnacle fair (2099 matches) ---


 tau  n_bets  n_matches    roi      lo     hi  claimed_edge  hit_rate
-inf    4198       2099 0.0011 -0.0082 0.0104        0.0009    0.5000
0.00    1852       1593 0.0150 -0.0297 0.0593        0.0248    0.4752
0.01    1250       1164 0.0237 -0.0371 0.0847        0.0343    0.4672
0.02     831        806 0.0367 -0.0404 0.1141        0.0442    0.4609
0.03     528        520 0.0835 -0.0187 0.1846        0.0557    0.4716
0.05     212        211 0.1325 -0.0375 0.3064        0.0809    0.4623



### Exploratory: the best arm (`continuity_lineup_loose_combo`)

New held-out set only (WP014); it has no old-set predictions. Same tests.

In [6]:
_, J_best = load_set(WP014 / 'cv_checkpoint_continuity_lineup_loose_combo.pkl', shared_new['windows'], 'new best arm', 100)
D_best = prep(J_best)
show(run_tests(J_best, D_best, 'new set, continuity_lineup_loose_combo', verbose=False), 5)

                                        test   n  estimate       lo       hi   half_1   half_2 result
Q1 log loss, Pinnacle+model - Pinnacle alone 995   0.00148  0.00086  0.00211  0.00213  0.00083 no hit
                    Q2 CLV per bet, tau=0.02 681  -0.02706 -0.03158 -0.02258 -0.02394 -0.03040 no hit
